# Can gene expression predict treatment condition?

This notebook answers the second seminar task: given a cell's gene expression profile, can we tell
which experimental condition it came from (`Control`, `IFNγ`, or `Co-culture`)?

All training happens in standalone, `sbatch`-submittable scripts under `classification/`
(`train_logreg.py`, `train_random_forest.py`, `train_xgboost.py`, `train_nn.py`), sharing common
feature/label preparation (`data_prep.py`) and evaluation code (`evaluate_utils.py`) so all four models
are trained/evaluated on **exactly the same** stratified train/test split. This notebook only *reads
back* their saved results (`classification/results/<model>/`) — it does not retrain anything, except
for the differential expression analysis in section 3, which is computed here directly.

**Task setup**
- Features: the 2000 highly-variable genes flagged during QC (`rna_filtered.h5ad`), log1p-normalized.
- Target: `perturbation_2` (3 classes: `Control`, `IFNγ`, `Co-culture`).
- Models: multinomial logistic regression, random forest, XGBoost, a small feed-forward neural network (PyTorch, CPU).
- Every model is evaluated on the **same held-out 20% test set**, plus 5-fold stratified CV on the training set for a stability check (not run for the NN — too costly to retrain 5x; it uses its own validation split for early stopping instead).

**Caveat to keep in mind throughout:** in this dataset, condition and technical batch are fully
confounded — each of the three conditions was profiled as its own separate experiment, with no shared
batch variable to control for. A high classification accuracy tells us the conditions are *separable*,
but does not by itself prove the separating genes are purely biology rather than partly a technical
signature. Section 1's discussion addresses this with two direct negative-control checks.

In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

RESULTS_DIR = "classification/results"
MODEL_DIRS = {
    "Logistic regression": "logreg",
    "Random forest": "random_forest",
    "XGBoost": "xgboost",
    "Neural network": "nn",
}
MODEL_COLORS = {  # fixed per model, reused across every plot in this notebook
    "Logistic regression": "#0072B2",
    "Random forest": "#009E73",
    "XGBoost": "#D55E00",
    "Neural network": "#CC79A7",
}

pd.set_option("display.max_colwidth", 200)

## 1. Test-set results per model

**Evaluation metrics, precisely defined** (all computed on the held-out 20% test set unless noted):

| Metric | Formula | What it captures |
|---|---|---|
| Accuracy | (# correct) / n | Overall fraction right; can be misleading under class imbalance. |
| Balanced accuracy | mean over classes of per-class recall | Same idea as accuracy but every class counts equally regardless of size. |
| Precision (per class) | TP / (TP + FP) | Of everything predicted class *c*, how much really was *c*. |
| Recall (per class) | TP / (TP + FN) | Of everything that really was *c*, how much did we catch. |
| F1 (per class) | 2·P·R / (P + R) | Harmonic mean of precision & recall. |
| Macro-F1 | unweighted mean of per-class F1 | Headline metric — treats all 3 classes equally despite their mildly different sizes (~57.6k / 73.1k / 87.6k cells). |
| Macro ROC-AUC (one-vs-rest) | for each class, area under the TPR-vs-FPR curve treating it as "this class vs. the rest" across every probability threshold; average the 3 | Threshold-independent measure of how well-separated the predicted probabilities are. |
| Confusion matrix | raw + row-normalized cross-tab of true × predicted | Shows *which* classes get confused with which. |
| 5-fold stratified CV macro-F1 (mean ± std) | refit on 5 different 80% slices of the training set, score each on its own held-out 20% | Checks the test score isn't a fluke of one split. |
| Majority-class baseline | accuracy/F1 of always predicting the largest class | The floor (40.1% accuracy, 0.19 macro-F1 here) any real model must clearly beat. |

For each model below: the per-class metrics table, the confusion matrix, and the ROC curve, all read
directly from its saved `metrics.json`/`.png` files. The neural network additionally shows its
training/validation loss curve, and its permutation importance — the *only* importance metric available
for it (unlike a linear model's coefficients or a tree ensemble's split gain, a multi-layer network's
weights don't map to a single feature's contribution, so there is no native alternative).

In [ ]:
def show_model_results(label, subdir):
    with open(f"{RESULTS_DIR}/{subdir}/metrics.json") as f:
        m = json.load(f)

    print(f"===== {label} =====")
    summary = {
        "accuracy": m["accuracy"], "balanced_accuracy": m["balanced_accuracy"],
        "macro_f1": m["macro_f1"], "macro_roc_auc": m["macro_roc_auc_ovr"],
    }
    if "cv" in m:
        summary["cv_macro_f1_mean"] = m["cv"]["cv_macro_f1_mean"]
        summary["cv_macro_f1_std"] = m["cv"]["cv_macro_f1_std"]
    display(pd.DataFrame([summary], index=[label]).round(4))

    report_df = pd.DataFrame(m["classification_report"]).T.round(3)
    display(report_df)

    display(Image(filename=f"{RESULTS_DIR}/{subdir}/confusion_matrix.png"))
    display(Image(filename=f"{RESULTS_DIR}/{subdir}/roc_curves.png"))

    if subdir == "nn":
        display(Image(filename=f"{RESULTS_DIR}/{subdir}/loss_curve.png"))
        perm_df = pd.read_csv(f"{RESULTS_DIR}/{subdir}/importances.csv").head(10)
        print("Top 10 genes by permutation importance (NN's only available importance metric):")
        display(perm_df)
    print()


for label, subdir in MODEL_DIRS.items():
    show_model_results(label, subdir)

**Discussion — how well does each model perform, and how do we know?**

All four models land in the same narrow band: **98.2–98.9% accuracy, macro-F1 0.98–0.99, macro
ROC-AUC ≥0.998**, all far above the 40.1% / 0.19 majority-class baseline. XGBoost edges out the others
(98.9% accuracy, macro-F1 0.989), with logistic regression a close second (98.6%) — a *linear* model
getting this close to the flexible tree ensembles suggests the three conditions are close to linearly
separable in this 2000-gene HVG space, so the extra model capacity buys only a small amount of
additional accuracy here.

CV macro-F1 (mean ± std, on the training set) matches the held-out test macro-F1 closely for every
model that has it (e.g. logreg: CV 0.9829 ± 0.0005 vs. test 0.9852) — a very tight std, so this is a
stable result, not a lucky split.

The confusion matrices show the *same* pattern for every model: the largest confusion by far is
**`Control ↔ IFNγ`** (e.g. XGBoost: 347 true-IFNγ cells predicted Control, 62 true-Control predicted
IFNγ), while `Co-culture` is almost never confused with either (≤23 cells either direction). That is
biologically coherent rather than suspicious: bulk IFNγ stimulation of a cell population typically
produces a **heterogeneous response** (a subset of "non/low-responder" cells that look transcriptionally
close to unstimulated controls — well documented in the IFN-response literature), whereas direct
immune-cell co-culture tends to produce a more complete, homogeneous activation signature. A pure
technical/batch artifact would not be expected to preferentially blur exactly the two conditions that
are mechanistically closest.

**Is ~98% suspiciously high?** Two negative-control experiments (run directly against the real
features/split) say no:
- **Label-shuffle test**: retraining logistic regression on the same features but randomly shuffled
  training labels collapses accuracy to 37.2% (macro-F1 0.26) — right at the chance/majority-baseline
  level. This rules out a leakage bug in the train/test split or feature pipeline.
- **Random-gene-subset test**: fitting the same classifier on *random*, unselected 50-gene subsets
  gives only 59–75% accuracy (3 trials) — well above chance but far below the ~98% achieved with
  informative genes, and far below the **98.4%** the top 50 *curated* genes alone achieve. A generic
  technical/batch confound tends to smear fairly evenly across genes, which would make random subsets
  nearly as predictive as curated ones — they aren't. Combined with the interferon/antigen-presentation
  biology recovered in section 3 below, the evidence points to real IFNγ/immune-co-culture biology, not
  a technical shortcut.
- **Residual caveat**: since condition and technical batch are structurally confounded by the
  experimental design, a purely technical contribution cannot be mathematically excluded with 100%
  certainty from expression data alone — the checks above make it much less likely, not impossible.

## 2. Model comparison

In [ ]:
rows = []
for label, subdir in MODEL_DIRS.items():
    with open(f"{RESULTS_DIR}/{subdir}/metrics.json") as f:
        m = json.load(f)
    rows.append({
        "model": label, "accuracy": m["accuracy"],
        "macro_f1": m["macro_f1"], "macro_roc_auc": m["macro_roc_auc_ovr"],
    })
comparison_df = pd.DataFrame(rows).set_index("model")
display(comparison_df.round(4))

metrics_to_plot = ["accuracy", "macro_f1", "macro_roc_auc"]
x = np.arange(len(comparison_df))
width = 0.25

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for i, metric in enumerate(metrics_to_plot):
    ax.bar(x + i * width, comparison_df[metric], width, label=metric)
ax.set_xticks(x + width)
ax.set_xticklabels(comparison_df.index, rotation=10)
ax.set_ylim(0.9, 1.0)  # zoomed in — all models are >90%, so a 0-1 axis would hide the real differences
ax.set_ylabel("score")
ax.set_title("Model comparison on the held-out test set")
ax.legend(frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

All four models clear 98% on every metric shown (axis starts at 0.9, not 0 — the majority-class
baseline of 40.1% accuracy / 0.19 macro-F1 sits far below this view; see section 1 for that number).
XGBoost is consistently the strongest, with the other three within about a percentage point of it and
of each other — a small, not a dramatic, spread.

## 3. Basic differential expression

As a model-free cross-check, independent of any classifier, we run a classical per-gene statistical
test directly here: `scanpy.tl.rank_genes_groups` with the **Wilcoxon rank-sum test**, one-vs-rest per
condition, on the same 2000 HVGs used for classification. For each gene and condition this gives:
- **score** — the Wilcoxon test statistic (scanpy's default gene ranking; more robust than raw
  fold-change at large sample sizes, see below).
- **logfoldchanges** — log2(mean expression in this condition / mean expression in the rest).
- **pvals_adj** — Benjamini-Hochberg-corrected p-value.

**A wrinkle at this sample size (~218k cells):** Wilcoxon p-values for real effects become so small
they underflow floating point to exactly 0 for a large fraction of genes — p-value stops being useful
for *ranking* the top genes (they're all tied at "immeasurably significant"). The volcano plots below
jitter those tied points for visibility and label genes by scanpy's `score` rank rather than by p-value
or raw fold-change (raw fold-change alone is noisy for genes with very low baseline counts).

In [ ]:
import scanpy as sc

adata = sc.read_h5ad("rna_filtered.h5ad")
adata_hvg = adata[:, adata.var["highly_variable"]].copy()
sc.tl.rank_genes_groups(adata_hvg, groupby="perturbation_2", method="wilcoxon")

de_df = pd.concat([
    sc.get.rank_genes_groups_df(adata_hvg, group=group).assign(group=group)
    for group in adata_hvg.obs["perturbation_2"].cat.categories
], ignore_index=True)
del adata, adata_hvg  # free the ~6GB object, not needed after this

In [ ]:
UP_COLOR, DOWN_COLOR, NS_COLOR = "#D55E00", "#0072B2", "#B3B3B3"
GROUPS = ["Control", "IFNγ", "Co-culture"]
PADJ_THRESH, LFC_THRESH, TOP_N_LABEL, Y_CEILING = 0.05, 1.0, 7, 300
LABEL_OFFSETS = [(4, 6), (4, -10), (4, 20), (4, -24), (4, 34), (4, -38), (4, 48)]
rng = np.random.RandomState(0)

fig, axes = plt.subplots(1, len(GROUPS), figsize=(15, 5.6), sharey=True)
for ax, group in zip(axes, GROUPS):
    sub = de_df[de_df["group"] == group].copy()
    sub["score_rank"] = np.arange(len(sub))  # rank_genes_groups_df rows are already sorted by score

    raw_neg_log10 = -np.log10(sub["pvals_adj"].clip(lower=10 ** -Y_CEILING))
    at_ceiling = raw_neg_log10 >= Y_CEILING
    jitter = np.where(at_ceiling, rng.uniform(-6, 6, size=len(sub)), 0.0)  # declutter only, not real variation
    sub["plot_y"] = raw_neg_log10.clip(upper=Y_CEILING) + jitter

    is_up = (sub["pvals_adj"] < PADJ_THRESH) & (sub["logfoldchanges"] > LFC_THRESH)
    is_down = (sub["pvals_adj"] < PADJ_THRESH) & (sub["logfoldchanges"] < -LFC_THRESH)
    is_ns = ~(is_up | is_down)

    ax.scatter(sub.loc[is_ns, "logfoldchanges"], sub.loc[is_ns, "plot_y"], s=6, c=NS_COLOR, alpha=0.5, linewidths=0)
    ax.scatter(sub.loc[is_up, "logfoldchanges"], sub.loc[is_up, "plot_y"], s=10, c=UP_COLOR, alpha=0.8, linewidths=0)
    ax.scatter(sub.loc[is_down, "logfoldchanges"], sub.loc[is_down, "plot_y"], s=10, c=DOWN_COLOR, alpha=0.8, linewidths=0)

    top_genes = sub[is_up].nsmallest(TOP_N_LABEL, "score_rank").sort_values("plot_y", ascending=False)
    for (_, row), offset in zip(top_genes.iterrows(), LABEL_OFFSETS):
        ax.annotate(row["names"], (row["logfoldchanges"], row["plot_y"]), fontsize=7.5,
                    xytext=offset, textcoords="offset points",
                    arrowprops=dict(arrowstyle="-", lw=0.5, color="#888888"))

    ax.axvline(LFC_THRESH, color="#999999", lw=0.7, ls="--")
    ax.axvline(-LFC_THRESH, color="#999999", lw=0.7, ls="--")
    ax.set_xlabel("log2 fold change")
    ax.set_title(group, pad=65)  # extra padding: labels on the topmost points sit above the axes
    ax.set_ylim(-10, Y_CEILING + 15)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("-log10(adjusted p-value)")
fig.suptitle("Differential expression: each condition vs. the rest")
plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.show()

**Discussion — biological plausibility of the top genes**

The `IFNγ` panel is dominated by textbook interferon-response genes: guanylate-binding proteins
(`GBP1/GBP2/GBP4`), `IDO1` and `IRF1` (hallmark IFNγ-response genes throughout the immuno-oncology
literature), and the chemokines `CXCL9/CXCL10/CXCL11` — literally named for their IFNγ-inducibility
("interferon gamma-induced protein 10" for CXCL10). The `Co-culture` panel features antigen-presentation
machinery (`HLA-B`, `CD74`, `HLA-DRA/DRB1/DPA1` — IFNγ is a well-known inducer of both MHC class I and
class II presentation) alongside stress genes (`HSPB1`, `TUBA1B`), consistent with a more complete
immune activation from direct cell contact — matching the confusion-matrix pattern from section 1
(`Co-culture` being the condition least confused with the others). The `Control` panel's top genes
(`LRRC17`, `PMEPA1`, `AKAP12`, ...) instead mark the *unstimulated* melanocyte state — i.e. what goes
*down* under IFNγ/co-culture.

No panel is dominated by housekeeping, mitochondrial, or ribosomal genes — the usual signature of a
technical/batch artifact is absent, reinforcing the "this is real biology" conclusion from section 1's
negative controls.

## Summary

1. **Yes** — gene expression predicts experimental condition with high confidence: 98.2–98.9% accuracy
   and macro-F1 0.98–0.99 across four independently-trained model families, all far above the 40.1%
   majority-class baseline, with tight cross-validation spread. Two negative controls (label-shuffling →
   chance-level accuracy; random gene subsets → far below curated-gene accuracy) rule out a pipeline
   leak or a purely diffuse technical/batch explanation.
2. **XGBoost performed best** (98.9% accuracy, macro-F1 0.989), narrowly ahead of logistic regression
   (98.6%), random forest (98.3%), and the neural network (98.2%). The gap between the simplest model
   and the most flexible is under 1 percentage point — the three conditions are close to linearly
   separable in this HVG space.
3. **Differential expression** independently recovers the same biology: canonical interferon-response
   and antigen-presentation genes (`GBP1/2/4`, `IDO1`, `IRF1`, `HLA-B/HLA-DRA/HLA-DRB1/HLA-DPA1`, `CD74`,
   `CXCL9/10/11`) mark `IFNγ` and `Co-culture`, while genes reflecting the unstimulated melanocyte state
   mark `Control`.
4. **Residual caveat**: condition and technical batch are structurally confounded in this dataset (each
   condition was profiled as its own separate experiment, with no shared batch variable in the
   metadata), so a purely technical contribution to the signal cannot be mathematically excluded with
   100% certainty from expression data alone. What we *can* say: the differentially expressed genes are
   specific and biologically coherent (matching well-established IFNγ/immune literature, not resembling
   typical technical artifacts), and random gene subsets do not approach the same classification
   accuracy — both weigh strongly against a technical explanation. Fully closing this would require an
   experimental design with an independent replicate or batch indicator per condition, which this
   dataset does not provide.